In [ ]:
import json
from collections import Counter
from datetime import datetime, timedelta
from itertools import product, zip_longest
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as goa
import regex
import requests
import yaml
from plotly.colors import qualitative, sample_colorscale
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

from datetime import datetime
import pytz
from src.utils import (
    guardarExcel,
    guardarExcelMulti
)

from datetime import timedelta
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
from src.utils import (
    isEmpty,
    loadEstaciones,
    loadLocalizaciones,
    localizeFecha,
    parallelizeFunction,
    rellenarId,
    removeDoubleQuotes,
    splitDataframe,
    getEstacionamientos,
    loadEstacionSinCTC
    
)
from src.api.api import GraylogAPIProcessor
from src.utils.util import loadEstaciones
from src.api.APIs import getCirculacionesPlanificadas

In [ ]:
from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime
)
from src.processor import SitraProcessor, MIEProcessor


In [ ]:

from reportlab.lib.pagesizes import letter, A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader
from pathlib import Path
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
from reportlab.lib.units import inch
from datetime import datetime
from reportlab.lib.colors import Color
from datetime import timedelta
from reportlab.platypus import PageBreak
# color24 = colors.qualitative.Dark24
# color12 = colors.qualitative.Set3

In [ ]:
# Tipos de tren que queremos
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "SUPRESIÓN",
    "End": "FIN",
    "Entry": "ENTRY",
    "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ORIGEN",
    "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    "Stopped": "STOP",
    "TrackingLost": "LOST_TRACK",
}

# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "PREVISIÓN",
            "APROXIMACIÓN",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ALTA",
            "SALIDA",
            "MANIOBRA_APROXIMACION"
            "MANIOBRA_LLEGADA",
            "MANIOBRA_SALIDA",
        ]
    )
}

In [ ]:
def cargarHistorico(
    start_date: str,
    end_date: str,
    estaciones: list[str],
    trenes: list[str],
    xSIV: bool = True,
    jCTC: bool = False,
    xREG: bool = False,
    pro: bool = True,
    maniobra:bool = True
):
    # Comprobamos que la fecha de fin sea después de la de inicio
    if end_date <= start_date:
        end_date = (pd.to_datetime(start_date) + timedelta(days=1)).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    historico = getHistoricoMOW(
        estaciones=estaciones,
        trenes=trenes,
        inicio=start_date,
        fin=end_date,
        xSIV=xSIV,
        jCTC=jCTC,
        xREG=xREG,
        pro=pro,
        maniobra= maniobra
    )
        
    if (xREG == False): 
        historico = historico[
            (historico["Fecha"] >= pd.to_datetime(start_date))
            & (historico["Fecha"] <= pd.to_datetime(end_date))
        ]
    else:
         historico = historico[
            (historico["FechaHora"] >= pd.to_datetime(start_date))
            & (historico["FechaHora"] <= pd.to_datetime(end_date))
        ]
    # Usamos movimientos auditados
    # historico = historico[
    #     np.invert(historico["FuenteVía"].isin(["PLANNED", "SITRA_PROVIDED"]))
    # ]
    
    
    
    if (xREG == False): 
        historico = historico[historico["NTécnico"].apply(isValidCode)].dropna(
            subset=["Movimiento"]
        )
        historico["mov_ord"] = historico["Movimiento"].apply(mov_sorter.get)
    return historico


In [ ]:
start_date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
end_date = datetime.now().strftime("%Y-%m-%d")

estaciones = []
xREG=True
ntrenes = [rellenarId(el) for el in np.arange(100000)]
# ntrenes = [rellenarId(el) for el in np.arange(2000, 6000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=False,
    jCTC=False,
    pro=True,
    xREG=xREG,
)
if (xREG == False):
    historico_pro = historico_pro.sort_values(
        by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
    ).reset_index(drop=True)

    # Añadir información de la fecha
    historico_pro["Día"] = historico_pro["Fecha"].dt.date
    historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
    historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
    historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:
xreg = historico_pro.copy()

In [ ]:
estaciones = []
xREG=False
ntrenes = [rellenarId(el) for el in np.arange(100000)]
# ntrenes = [rellenarId(el) for el in np.arange(2000, 6000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=True,
    jCTC=False,
    pro=True,
    xREG=xREG,
)
if (xREG == False):
    historico_pro = historico_pro.sort_values(
        by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
    ).reset_index(drop=True)

    # Añadir información de la fecha
    historico_pro["Día"] = historico_pro["Fecha"].dt.date
    historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
    historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
    historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:
xreg = xreg[
        (xreg["FechaHora"] >= pd.to_datetime(start_date))
        & (xreg["FechaHora"] <= pd.to_datetime(end_date))
    ].copy()

In [ ]:
supresiones = historico_pro[historico_pro["Movimiento"] == "ELIMINACIÓN"]

In [ ]:
no_comercial =["Mercancias","Material Vacio","T.L.E.","Transporte excepcional","Maquina Aislada Mercancias","MAQUINA AISLADA","Material vacio RAM"]

In [ ]:
supresiones = supresiones[~supresiones["Producto"].isin(no_comercial)].copy()

In [ ]:
supresiones["Producto"].unique()

<h3>Supresiones en Origen</h3>

In [ ]:
sub_dfs = [grupo for _, grupo in supresiones.groupby("NTécnico")]

In [ ]:
supresiones_origen = [
    df
    for df in sub_dfs
    if (df["Secuencia"] == 1).any()
]

In [ ]:
def timedelta_to_hhmmss(td):
    if pd.isna(td):
        return None  # o "" o "0:00:00", como prefieras

    total_seconds = int(td.total_seconds())
    sign = "-" if total_seconds < 0 else ""
    total_seconds = abs(total_seconds)

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60

    return f"{sign}{hours:02d}:{minutes:02d}:{seconds:02d}"

In [ ]:
origen = []
for df in supresiones_origen:
    df = df[["FechaOrigen","NTécnico","Producto","Código","Nombre","SalidaPlanificada","Fecha"]].copy()
    df.rename(columns={"Fecha":"FechaSupresión"}, inplace=True)
    df["Antelación"] = df["SalidaPlanificada"] - df["FechaSupresión"]
    df["Antelación"] = df["Antelación"].apply(timedelta_to_hhmmss)
    origen.append(df)
    

In [ ]:
origenes = pd.concat(origen,ignore_index=True)


In [ ]:
subdireciones = pd.read_csv("data/Subdirección.csv")


subdireciones['Subdirección'] = subdireciones['Subdirección'].str.replace(r'^RC', 'SD', regex=True)

subdireciones['Subdirección'] = subdireciones['Subdirección'].replace(
    'RED DE ALTA VELOCIDAD (RAV)', 'SD AV'
)

In [ ]:
origenes = origenes.merge(
    subdireciones,
    how = "left",
    on= "Código",

)

In [ ]:
origenes = origenes[["Subdirección","Código","Nombre","NTécnico","Producto","SalidaPlanificada","FechaSupresión","Antelación"]]

In [ ]:
origenes

<h3>Supresiones Cambio destino</h3>

In [ ]:
no_origen = supresiones[~supresiones["NTécnico"].isin(origenes["NTécnico"])].copy()

In [ ]:
destino = xreg[xreg["TipoMovimiento"] == "XREG_FORECAST_CHANGE_DESTINATION"].copy()

In [ ]:
interrupcion = xreg[xreg["TipoMovimiento"] == "XREG_FORECAST_SECTION_INTERRUPTION"].copy()

In [ ]:
no_origen_clear = no_origen[["FechaOrigen","NTécnico","Código","Nombre","SalidaPlanificada","Secuencia","Producto"]].copy()

In [ ]:
start_date = pd.to_datetime(start_date)
mask = no_origen_clear["SalidaPlanificada"].dt.date == (start_date - pd.Timedelta(days=1)).date()

In [ ]:
no_origen_clear.loc[mask, "SalidaPlanificada"] = no_origen_clear.loc[mask, "SalidaPlanificada"].apply(
    lambda x: start_date + pd.Timedelta(hours=x.hour, minutes=x.minute, seconds=x.second)
)

In [ ]:
suprimido_destinos = pd.merge(
    no_origen_clear,
    destino[["FechaHora","NTécnico","Nombre","Código","SecuenciaFin"]],
    left_on=["NTécnico","Secuencia"],
    right_on =["NTécnico","SecuenciaFin"],
    how = "left"
)

In [ ]:
suprimido_destinos_1 = suprimido_destinos.dropna(subset=["SecuenciaFin"]).copy()

In [ ]:
suprimido_destinos_1.drop(columns=["Código_y","Nombre_y","SecuenciaFin"],inplace = True)

In [ ]:
suprimido_destinos_1.rename (columns={"Código_x":"Código","Nombre_x":"Nombre","FechaHora":"HoraAviso"},inplace=True)

In [ ]:
suprimido_destinos_1["Incidencia"] = "Cambio destino"

In [ ]:
suprimido_destinos_1 = suprimido_destinos_1[["Código","Nombre","NTécnico","Producto","Incidencia","SalidaPlanificada","HoraAviso"]]

In [ ]:
suprimido_destinos_1["Antelación"]  = suprimido_destinos_1["SalidaPlanificada"]-suprimido_destinos_1["HoraAviso"]

In [ ]:
suprimido_destinos_1["Antelación"] = suprimido_destinos_1["Antelación"].apply(timedelta_to_hhmmss)

In [ ]:
suprimido_destinos_1.reset_index(drop=True,inplace=True)

In [ ]:
suprimido_interrupcion = pd.merge(
    no_origen_clear,
    interrupcion[["NTécnico","FechaHora","CódigoIncio"]],
    left_on=["NTécnico","Código"],
    right_on=["NTécnico","CódigoIncio"],
    how = "left"
)

In [ ]:
suprimido_interrupcion_1 = suprimido_interrupcion[~suprimido_interrupcion["FechaHora"].isna()].copy()


In [ ]:
suprimido_interrupcion_1["Antelación"] = suprimido_interrupcion_1["SalidaPlanificada"]- suprimido_interrupcion_1["FechaHora"]

In [ ]:
suprimido_interrupcion_1["Antelación"] = suprimido_interrupcion_1["Antelación"].apply(timedelta_to_hhmmss)

In [ ]:
suprimido_interrupcion_1["Incidencia"] = "Interrupción"

In [ ]:
suprimido_interrupcion_1 = suprimido_interrupcion_1[["Código", "Nombre","NTécnico","Producto","Incidencia","SalidaPlanificada","FechaHora","Antelación"]].rename(columns={"FechaHora":"HoraAviso"}).copy()

In [ ]:
supresiones = xreg[xreg["TipoMovimiento"] == "XREG_SUPPRESSION"].copy()

In [ ]:
sin_aviso = supresiones[(~supresiones["NTécnico"].isin(destino["NTécnico"])) & (~supresiones["NTécnico"].isin(interrupcion["NTécnico"])) & (supresiones["Secuencia"] != 1)]

In [ ]:
sin_aviso_1 = no_origen[(~no_origen["NTécnico"].isin(destino["NTécnico"])) & (~no_origen["NTécnico"].isin(interrupcion["NTécnico"]))]

In [ ]:
sin_aviso_2 = pd.merge(
    sin_aviso[["FechaHora","NTécnico","Código"]],
    sin_aviso_1[["NTécnico","Código","Nombre","Producto","SalidaPlanificada"]],
    on = ["NTécnico","Código"],
    how = "left"
)

In [ ]:
sin_aviso_total = sin_aviso_2[~sin_aviso_2["SalidaPlanificada"].isna()].copy()

In [ ]:
sin_aviso_total.reset_index(drop=True,inplace=True)

In [ ]:
sin_aviso_total["Incidencia"] = "Cambio destino"

In [ ]:
sin_aviso_total["HoraAviso"] = "No anunciada"

In [ ]:
sin_aviso_total["Antelación"] = sin_aviso_total["SalidaPlanificada"] - sin_aviso_total["FechaHora"]

In [ ]:
sin_aviso_total["Antelación"] =sin_aviso_total["Antelación"].apply(timedelta_to_hhmmss)

In [ ]:
sin_aviso_total = sin_aviso_total[["Código","Nombre","NTécnico","Producto","Incidencia","SalidaPlanificada","HoraAviso","FechaHora","Antelación"]].rename(columns={"FechaHora":"HoraSupresión"})

In [ ]:
tabla2 = pd.concat(
    [suprimido_destinos_1, suprimido_interrupcion_1, sin_aviso_total],
    ignore_index=True
)

In [ ]:
tabla2 = tabla2.merge(
    subdireciones,
    on= "Código",
    how = "left"
)

In [ ]:
tabla2 = tabla2[["Subdirección","Código","Nombre","NTécnico","Producto","Incidencia","SalidaPlanificada","HoraAviso","HoraSupresión","Antelación"]]

<h3> Cambio Origen </h3>

In [ ]:
cambio_origen = xreg[xreg["TipoMovimiento"] == "XREG_FORECAST_CHANGE_ORIGIN"].copy()

In [ ]:
guadiana = xreg[xreg["TipoMovimiento"] == "XREG_GUADIANA_DEPARTURE"]

In [ ]:
cambio_origen_1 = cambio_origen[cambio_origen["NTécnico"].isin(guadiana["NTécnico"])].copy()


In [ ]:
cambio_origen_2 = cambio_origen[~cambio_origen["NTécnico"].isin(guadiana["NTécnico"])].copy()

In [ ]:
cambio_origen_2

In [ ]:
interrupcion_1 = interrupcion[interrupcion["NTécnico"].isin(guadiana["NTécnico"])].copy()

In [ ]:
interrupcion_2 = interrupcion[~interrupcion["NTécnico"].isin(guadiana["NTécnico"])].copy()

In [ ]:
cambio_origen_1["TieneGuadiana"] = True
cambio_origen_2["TieneGuadiana"] = False
interrupcion_1["TieneGuadiana"] = True
interrupcion_2["TieneGuadiana"] = False
cambio_origen_1["Incidencia"] = "Cambio origen"
cambio_origen_2["Incidencia"] = "Cambio origen"
interrupcion_1["Incidencia"] = "Interrupción"
interrupcion_2["Incidencia"] = "Interrupción"

In [ ]:
cambio_origen_1 = cambio_origen_1[["CódigoIncio","NTécnico","Incidencia","TieneGuadiana"]].rename(columns={"CódigoIncio":"Código"}).copy()
cambio_origen_2 = cambio_origen_2[["CódigoIncio","NTécnico","Incidencia","TieneGuadiana"]].rename(columns={"CódigoIncio":"Código"}).copy()
interrupcion_1 = interrupcion_1[["CódigoFin","NTécnico","Incidencia","TieneGuadiana"]].rename(columns={"CódigoFin":"Código"}).copy()
interrupcion_2 = interrupcion_2[["CódigoFin","NTécnico","Incidencia","TieneGuadiana"]].rename(columns={"CódigoFin":"Código"}).copy()

In [ ]:
dfs = [cambio_origen_1, cambio_origen_2, interrupcion_1, interrupcion_2]

Guadiana = pd.concat(dfs, ignore_index=True)

In [ ]:
historico_pro_comercial = historico_pro[~historico_pro["Producto"].isin(no_comercial)].copy()

In [ ]:
Guadiana1 = pd.merge(
    Guadiana,
    historico_pro_comercial[["NTécnico","Código","Nombre","Producto"]],
    on  = ["NTécnico","Código"],
    how = "left"
)

In [ ]:
estaciones = loadEstaciones()
estaciones_sinctc = loadEstacionSinCTC()
estaciones = pd.concat([estaciones,estaciones_sinctc],ignore_index=True)

In [ ]:
regex = r'\s+(RAM|AV|RC)$'
estaciones['Nombre'] = estaciones['Nombre'].str.replace(regex, '', regex=True)
estaciones.drop_duplicates(subset="Código",inplace=True)

In [ ]:
Guadiana1 = pd.merge(
    Guadiana1,
    estaciones[["Código","Nombre"]],
    on = "Código",
    how = "left"
)

In [ ]:
Guadiana1 = Guadiana1.merge(
    subdireciones,
    on ="Código",
    how = "left"
)

In [ ]:
Guadiana1 = Guadiana1[["Subdirección","Código","Nombre_y","NTécnico","Producto","Incidencia","TieneGuadiana"]].rename(columns = {"Nombre_y":"Nombre"})

In [ ]:
Guadiana1

In [ ]:
# cambio_origen2 = pd.merge(
#     cambio_origen_1[["FechaHora","NTécnico","CódigoIncio"]],
#     historico_pro_comercial[["NTécnico","Código","Nombre","Producto","SalidaPlanificada"]],
#     left_on=["NTécnico","CódigoIncio"],
#     right_on=["NTécnico","Código"],
#     how = "left"
# )

In [ ]:
# cambio_origen2["Incidencia"] ="Cambio origen"

In [ ]:
# cambio_origen2 = cambio_origen2[["Código","Nombre","Producto","Incidencia","SalidaPlanificada","FechaHora"]].rename(columns={"FechaHora":"HoraAviso"})

In [ ]:
# cambio_origen2["Antelación"] = None

In [ ]:
# cambio_origen_no_aviso = guadiana[(~guadiana["NTécnico"].isin(cambio_origen["NTécnico"])) & (~guadiana["NTécnico"].isin(interrupcion["NTécnico"]))]

In [ ]:
# cambio_origen3 = pd.merge(
#     cambio_origen_no_aviso[["FechaHora","NTécnico","Código"]],
#     historico_pro_comercial[["NTécnico","Código","Nombre","Producto","SalidaPlanificada"]],
#     left_on=["NTécnico","Código"],
#     right_on=["NTécnico","Código"],
#     how = "left"
# )

In [ ]:
# cambio_origen3["Incidencia"] = "Cambio origen"

In [ ]:
# cambio_origen3 = cambio_origen3[["Código","Nombre","Producto","Incidencia","SalidaPlanificada","FechaHora"]].rename(columns={"FechaHora":"HoraSalidaGuadiana"})
# cambio_origen3["Antelación"] = None

In [ ]:
# interupcion_con_aviso = interrupcion[interrupcion["NTécnico"].isin(guadiana["NTécnico"])]

In [ ]:
# interupcion_con_aviso_1 = pd.merge(
#     interupcion_con_aviso[["FechaHora","NTécnico","CódigoFin"]],
#     historico_pro_comercial[["NTécnico","Código","Nombre","Producto","SalidaPlanificada"]],
#     left_on= ["NTécnico","CódigoFin"],
#     right_on= ["NTécnico","Código"],
#     how = "left"
# )

In [ ]:
# interupcion_con_aviso_1["Incidencia"] = "Interrupción"

In [ ]:
# interupcion_con_aviso_1 = interupcion_con_aviso_1[["Código","Nombre","Producto","Incidencia","SalidaPlanificada","FechaHora"]].rename(columns={"FechaHora":"HoraAviso"})
# interupcion_con_aviso_1["Antelación"] = None

In [ ]:
# tabla3 = pd.concat([cambio_origen2,interupcion_con_aviso_1,cambio_origen3],ignore_index=True)

In [ ]:
# tabla3 = tabla3[["Código","Nombre","Producto","Incidencia","SalidaPlanificada","HoraAviso","HoraSalidaGuadiana","Antelación"]]

In [ ]:
# tabla3["Producto"].unique()

<h3> Supresiones Destino</h3>

In [ ]:
lista_subdfs = [group for _, group in historico_pro_comercial.groupby(['NTécnico', 'FechaOrigen'])]
destino = []

# Recorremos cada sub-DataFrame
for sub_dfs in lista_subdfs:
    sub_dfs_sorted = sub_dfs.sort_values(by='Secuencia', ascending=True)
    ultima_fila = sub_dfs_sorted.iloc[-1]
    if ultima_fila['Código'] == ultima_fila['CódigoDestino']:
        destino.append(ultima_fila)

# Mostramos la lista de últimas secuencias
df_destino = pd.DataFrame(destino)
df = df_destino[(df_destino["Movimiento"] == "ELIMINACIÓN") & (df_destino["Secuencia"] > 1)]

In [ ]:
if df.empty:
    data = {
        "Supresiones_origen":origenes,
        "Supresiones_cambio_destino":tabla2,
        "Guadiana":Guadiana1,
    }
else:
    data = {
        "Supresiones_origen":origenes,
        "Supresiones_cambio_destino":tabla2,
        "Guadiana":Guadiana1,
        "SupresionesDestino":df
    }
fecha_ayer = datetime.now() - timedelta(days=1)
semana= fecha_ayer.strftime('%U')
fecha_ayer_str = fecha_ayer.strftime('%Y-%m-%d')
nuevo_nombre = f"Semana{semana}_{fecha_ayer_str}_Supresiones.xlsx"
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual") / nuevo_nombre


In [ ]:
guardarExcelMulti(data,fname)

In [ ]:
def obtener_estado(anticipacion):
    if anticipacion > pd.Timedelta(minutes=5):
        return "más de 5 minutos"
    elif pd.Timedelta(minutes=0) < anticipacion <= pd.Timedelta(minutes=5):
        return "entre 5 minutos"
    else:
        return "sin anticipación"


In [ ]:
import plotly.express as px
import pandas as pd

# Contar cantidad por estado y crear DataFrame
def  gráfica_origenes(df):
    if df.empty:
        return 
    df['Antelación'] = pd.to_timedelta(df['Antelación'], errors='coerce')
    df['Estado'] = df['Antelación'].apply(obtener_estado)
    conteo_estado = df['Estado'].value_counts().reset_index()
    conteo_estado.columns = ['Estado', 'Cantidad']

    # Crear gráfica de torta tipo donut más grande
    fig = px.pie(
        conteo_estado,
        values='Cantidad',
        names='Estado',
        hole=0.25,  # donut más grande (hueco más pequeño)
        color='Estado',
        color_discrete_map={
            "sin anticipación": "#C91D1D",     # rojo
            "entre 5 minutos": "#CD8F1A",      # naranja
            "más de 5 minutos": "#3DC93D"      # verde
        }
    )

    # Mostrar valores y porcentajes dentro del donut y bordes blancos
    fig.update_traces(
        textposition='inside',
        textinfo='value+percent',  # muestra etiqueta, valor y porcentaje
        marker=dict(line=dict(color='white', width=2))
    )

    # Ajustar título centrado vertical y horizontalmente
    fig.update_layout(
        title={
            'text': "Distribución de la Anticipación en supresiones en el origen",
            'x': 0.5,       # centrar horizontal
            'y': 0.95,      # ajustar vertical (más cerca del centro del gráfico)
            'xanchor': 'center',
            'yanchor': 'top'
        },
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=-0.1,
            xanchor='center',
            x=0.5,
            font=dict(size=14)
        ),
        height=500
    )
    return fig


In [ ]:
# --- Clasificación de la Incidencia (Columna X en el gráfico) ---
def clasificar_incidencia(row):
    if row['Incidencia'] == 'Cambio destino' and row['HoraAviso'] == 'No anunciada':
        return 'No avisado (Cambio destino)'
    elif row['Incidencia'] == 'Cambio destino':
        return 'Cambio destino'
    else:
        # En el caso de que tuvieras otras incidencias como 'Interrupción'
        return row['Incidencia']



In [ ]:
test = tabla2[tabla2["Subdirección"] == "SD CENTRO"].copy()

In [ ]:
test

In [ ]:
test['Incidencia_Clasificada'] = test.apply(clasificar_incidencia, axis=1)

In [ ]:
test

In [ ]:
def grafica_destino(df):
    # Asegúrate de que 'Antelación' sea de tipo timedelta
    df['Antelación'] = pd.to_timedelta(df['Antelación'], errors='coerce')
    
    # Clasificar el estado basado en 'Antelación'
    df['Estado'] = df['Antelación'].apply(obtener_estado)
    
    # Clasificar las incidencias
    df['Incidencia_Clasificada'] = df.apply(clasificar_incidencia, axis=1)
    
    # Agrupar los datos por la columna 'Incidencia_Clasificada' y contar las incidencias
    df_grouped = df.groupby(['Incidencia_Clasificada', 'Estado']).size().reset_index(name='Cuenta')
    
    # Generar la gráfica
    fig = px.bar(
        df_grouped,
        x='Incidencia_Clasificada',   # Categorías en eje X
        y='Cuenta',                   # Valores de las barras
        color='Estado',               # Variable para color/apilamiento
        orientation='v',              # Barras verticales (usa 'h' para horizontales)
        title='Anticipación de supresión correspondiente a cambio de destino e Interrupción',
        labels={
            'Cuenta': 'Número de Incidencias',
            'Incidencia_Clasificada': 'Tipo de Incidencia',
            'Estado': 'Anticipación del Aviso'
        },
        category_orders={
            "Estado": ['sin anticipación', 'entre 5 minutos', "más de 5 minutos"]
        },
        color_discrete_map={
            'sin anticipación': 'red',
            'entre 5 minutos': 'orange',
            'más de 5 minutos': 'green'
        }
    )

    # Ajustes visuales de la gráfica
    fig.update_layout(
        xaxis_title='Tipo de Incidencia',
        yaxis_title='Número de Incidencias',
        legend_title='Anticipación',
        height=600,
        width=900,
        title_x=0.5,        # Centra el título
        bargap=0.2,         # Reduce el espacio entre grupos de barras
        bargroupgap=0.1     # Reduce el espacio entre barras apiladas
    )

    # Mostrar la gráfica
    return fig

In [ ]:
def gráficaGuadiana(df):
    conteo_guadiana = df.groupby(['Incidencia', 'TieneGuadiana']).size().reset_index(name='Cantidad')
    total_por_incidencia = df.groupby('Incidencia').size().reset_index(name='Total')
    df_pct = pd.merge(conteo_guadiana, total_por_incidencia, on='Incidencia')
    df_pct['Porcentaje'] = df_pct['Cantidad'] / df_pct['Total'] * 100
    fig = px.bar(
        df_pct,
        x='Incidencia',
        y='Porcentaje',
        color='TieneGuadiana',
        barmode='group',  # barras lado a lado
        labels={
            'Incidencia': 'Tipo de Incidencia',
            'Porcentaje': 'Porcentaje (%)',
            'TieneGuadiana': 'Tiene Guadiana'
        },
        title='Salidas Por Guadiana',
        color_discrete_map={True: 'green', False: 'red'}
    )

    fig.update_layout(
        yaxis=dict(range=[0, 100]),  # rango fijo 0-100%
        title_x=0.5,
        height=600,
        width =900,
        bargap=0.2,         # Reduce el espacio entre grupos de barras
        bargroupgap=0.1     # Reduce el espacio entre barras apiladas
    )


    return fig


In [ ]:
from datetime import datetime, timedelta
from pathlib import Path
from reportlab.lib.units import inch

def add_header(canvas, doc):
    canvas.saveState()
    fecha_actual = (datetime.now() - timedelta(days=1)).strftime("%d-%m-%Y")

    # Logo
    try:
        logo_path = Path("data/logo.png")
        canvas.drawImage(str(logo_path), doc.leftMargin, doc.height + doc.topMargin - 0.75*inch,
                         width=2*inch, height=0.75*inch, preserveAspectRatio=True)
    except:
        canvas.setFont("Helvetica", 10)
        canvas.drawString(doc.leftMargin, doc.height + doc.topMargin - 0.5*inch, "[LOGO NO ENCONTRADO]")

    # Posición base del título
    center_x = doc.width / 2.0 + doc.leftMargin
    y = doc.height + doc.topMargin - 0.4*inch

    # Títulos centrados
    canvas.setFont("Helvetica-Bold", 10)
    canvas.drawCentredString(center_x, y, "INDICADORES CALIDAD MSE")
    canvas.drawCentredString(center_x, y - 12, "Anticipación de Supresiones (Intervenciones)")
    canvas.drawCentredString(center_x, y - 24, f"Análisis de datos {fecha_actual}")

    # Fecha a la derecha
    canvas.setFont("Helvetica", 10)
    canvas.drawRightString(doc.width + doc.leftMargin, doc.height + doc.topMargin - 0.3*inch,
                           f"Fecha: {fecha_actual}")

    canvas.restoreState()


In [ ]:
def add_footer(canvas, doc):
        canvas.saveState()
        
        # Obtener fecha actual
        fecha = (datetime.now() - timedelta(days=1)).strftime("%d-%m-%Y")
        styles = getSampleStyleSheet()
        
        # Configurar pie de página
        footer_text_left = "SD. de Sistemas y Medios Operacionales<br/>D. de Circulación y Gestión de Capacidad<br/>DG. de OPERACIONES Y EXPLOTACIÓN"
        footer_text_center = f"Página {doc.page}"
        footer_text_right = f"Fecha: {fecha}"
        footer_style = ParagraphStyle(
        'Footer',
        parent=styles['Normal'],
        fontSize=7,
        leading=10,
        spaceBefore=5,
        alignment=0  # Alineación izquierda
    )
        left_paragraph = Paragraph(footer_text_left, footer_style)
        left_paragraph.wrapOn(canvas, 3.5*inch, 0.5*inch)
        left_paragraph.drawOn(canvas, 0.5*inch, 0.3*inch)
        canvas.setFont("Helvetica", 9)
    
        
        # Centro (calculamos la posición central)
        text_width = canvas.stringWidth(footer_text_center, "Helvetica", 9)
        canvas.drawString((doc.width + doc.leftMargin + doc.rightMargin - text_width) / 2, 
                          0.5*inch, footer_text_center)
        
        # Derecha
        right_pos = doc.width + doc.leftMargin - canvas.stringWidth(footer_text_right, "Helvetica", 9) - 0.75*inch
        canvas.drawString(right_pos, 0.5*inch, footer_text_right)
        
        # Línea separadora
        canvas.setStrokeColor(colors.gray)
        canvas.setLineWidth(0.5)
        # canvas.line(0.75*inch, 0.7*inch, doc.width + doc.leftMargin - 0.75*inch, 0.7*inch)
        
        canvas.restoreState()

In [ ]:
def add_header_footer(canvas, doc):
    add_header(canvas, doc)
    add_footer(canvas, doc)

In [ ]:
dirección =["SD CENTRO","SD SUR","SD NORTE","SD NOROESTE","SD NORESTE", "SD ESTE", "SD AV"]

In [ ]:
import os
from datetime import datetime, timedelta
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.units import inch
from reportlab.lib.colors import Color
from io import BytesIO
from plotly.io import write_image
bookmarks = {}


def Supresiones_PDF():
    """Versión con encabezado usando tabla e incluyendo imagen logo"""
    fecha_ayer = datetime.now() - timedelta(days=1)
    fecha_formateada = fecha_ayer.strftime("%Y-%m-%d")
    carpeta_destino = r"C:\Users\xiangzhou.zhang\Documents\Data"
    os.makedirs(carpeta_destino, exist_ok=True)
    semana = fecha_ayer.isocalendar()[1]
    filename = os.path.join(carpeta_destino, f"Semana{semana}_{fecha_formateada}_supresiones_3.pdf")

    doc = SimpleDocTemplate(filename, 
                            pagesize=A4,        
                            leftMargin=0.75*inch,
                            rightMargin=0.75*inch,
                            topMargin=1*inch,
                            bottomMargin=1*inch)
    
    styles = getSampleStyleSheet()
    story = []

    fecha_ayer = (datetime.now() - timedelta(days=1)).strftime("%d-%m-%Y")

    titulo_style = ParagraphStyle(
        'CustomTitle',
        parent=styles['Heading1'],
        fontSize=15,
        spaceAfter=20,
        alignment=1,
        textColor=colors.black
    )
    story.append(Spacer(1, 70))
    
    verde_oscuro = Color(0/255, 100/255, 0/255)
    verde_claro = Color(52/255, 207/255, 145/255) 
    barra_contenido = [
        ['ÍNDICE DE CONTENIDO']
    ]
    barra_descipción = [
        ['DESCRIPCIÓN']
    ]
    estilo_indice = TableStyle([
        ('BACKGROUND', (0,0), (-1,0), verde_oscuro),  
        ('TEXTCOLOR', (0,0), (-1,0), colors.white),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),  # Cambiado a negrita
        ('FONTSIZE', (0,0), (-1,0), 12),
        ('ALIGN', (0,0), (-1,-1), 'CENTER'),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('BOX', (0,0), (-1,-1), 1, colors.black),  
        ('INNERGRID', (0,0), (-1,-1), 0.5, colors.grey), 
        ('LEFTPADDING', (0,0), (-1,-1), 100),
        ('RIGHTPADDING', (0,0), (-1,-1), 100),
    ])
    estilo_descripción = TableStyle([
        ('BACKGROUND', (0,0), (-1,0), verde_claro),  
        ('TEXTCOLOR', (0,0), (-1,0), colors.white),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTSIZE', (0,0), (-1,0), 9),
        ('ALIGN', (0,0), (-1,-1), 'CENTER'),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('BOX', (0,0), (-1,-1), 1, colors.black),  
        ('INNERGRID', (0,0), (-1,-1), 0.5, colors.grey),  
        ('LEFTPADDING', (0,0), (-1,-1), 100),
        ('RIGHTPADDING', (0,0), (-1,-1), 100),
    ])
    contenido_style = ParagraphStyle(
        'Contenido',
        parent=styles['Normal'],
        spaceAfter=12
    )
    contenido_center = ParagraphStyle(
        'ContenidoCenter',
        parent=styles['Normal'],
        alignment=1,  # 1 = center
        spaceAfter=12
    )
 
    descripcion = Table(barra_descipción, colWidths=[6*inch])
    descripcion.setStyle(estilo_indice) 
    story.append(descripcion)   
    story.append(Spacer(1, 30))
    story.append(Paragraph(    "El presente informe tiene como objetivo evaluar la anticipación con la que se reciben las supresiones derivadas de peticiones de intervención, "
    "tales como cambios de origen, cambios de destino e interrupciones entre dos puntos intermedios del recorrido.<br />"
    "Para ello, se analizarán cuatro casos específicos:<br />"
    "1. Supresiones en la dependencia  de origen.<br />"
    "2. Supresiones ocasionadas por cambios de destino e interrupción entre dos puntos.<br />"
    "3. Salida guadiana derivados a cambio de origen e interrupción entre dos puntos.<br />"
    "4. Supresiones en la dependencia destino.", contenido_style))
    story.append(Spacer(1, 30))
    titulo_indice_style = ParagraphStyle(
        'TituloIndice',
        parent=styles['Normal'],
        fontSize=12,
        textColor=colors.white,
        alignment=1,
        fontName='Helvetica-Bold'  # Asegurado que esté en negrita
    )
    sections_style = ParagraphStyle(
    'SectionsStyle',
    parent=styles['Normal'],
    fontSize=12,
    textColor=colors.blue,  # azul para simular enlace
    alignment=0,  # alineado a la izquierda
    fontName='Helvetica-Bold',
    spaceAfter=6
    )

    link_style = ParagraphStyle(
        'LinkStyle',
        parent=styles['Normal'],
        fontSize=12,
        textColor=colors.white,
        alignment=1,
        # fontName='Helvetica-Bold'
    )
    titulo_indice_table = Table(
    [[Paragraph('ÍNDICE DE CONTENIDO', titulo_indice_style)]],
    colWidths=[6*inch]
    )
    titulo_indice_table.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,-1), verde_oscuro),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,-1), 'Helvetica-Bold'),
    ('FONTSIZE', (0,0), (-1,-1), 12),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
    ('BOX', (0,0), (-1,-1), 1, colors.black),
    ('TOPPADDING', (0,0), (-1,-1), 2),
    ('BOTTOMPADDING', (0,0), (-1,-1), 2),
    ]))

    story.append(titulo_indice_table)
    story.append(Spacer(1, 12))
    sections_style = ParagraphStyle(
    'SectionsStyle',
    parent=styles['Normal'],
    fontSize=12,
    textColor=colors.blue,  # color azul para simular enlaces
    alignment=0,  # alineado a la izquierda
    fontName='Helvetica-Bold',
    spaceAfter=6
    )

    sections_data = [
        'Análisis de Anticipación SD CENTRO',
        'Análisis de Anticipación de vía SD SUR',
        'Análisis de Anticipación de vía SD NORTE',
        'Análisis de Anticipación de vía SD NOROESTE',
        'Análisis de Anticipación de vía SD NORESTE',
        'Análisis de Anticipación de vía SD ESTE',
        'Análisis de Anticipación de vía SD AV',
    ]
    # Agregar enlaces al índice usando Paragraph
    for i, section_name in enumerate(sections_data, 1):
        link_text = f"<link href='#section_{i}' color='black'>{section_name}</link>"
        story.append(Paragraph(link_text, contenido_style))

    # # Estilo para el índice
    # indice_style = TableStyle([
    #     ('BACKGROUND', (0,0), (-1,0), verde_oscuro),
    #     ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    #     ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),  # Mantener negrita
    #     ('FONTSIZE', (0,0), (-1,0), 12),
    #     ('ALIGN', (0,0), (-1,-1), 'LEFT'),
    #     ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
    #     ('BOX', (0,0), (-1,-1), 1, colors.black),
    #     ('INNERGRID', (0,0), (-1,-1), 0.5, colors.grey),
    #     ('LEFTPADDING', (0,0), (-1,-1), 50),
    #     ('RIGHTPADDING', (0,0), (-1,-1), 50),
    #     ('TOPPADDING', (0,0), (-1,-1), 8),
    #     ('BOTTOMPADDING', (0,0), (-1,-1), 8),
    # ])
    
    # # Agregar la tabla de índice con hipervínculos
    # contenido_table = Table(indice_contenido, colWidths=[6*inch])
    # contenido_table.setStyle(indice_style)
    
    # story.append(Spacer(1, 30))
    # story.append(contenido_table)

    story.append(PageBreak())
    # Indicadores
    story.append(Spacer(1,70))
    barra_indicadores = [["INDICADORES"]]
    indicador = Table(barra_indicadores, colWidths=[6 * inch])
    indicador.setStyle(estilo_indice)
    story.append(indicador)
    # story.append(Indicadores)
    story.append(Spacer(1, 30))
    story.append(Paragraph("1. Distribución de la Anticipación en supresiones del Punto de origen: Gráfica donde muestra el grado de anticipación", contenido_style))
    story.append(Paragraph("2. Anticipación de supresiones por cambio de destino y su nivel de anticipación", contenido_style))
    story.append(Paragraph("2. Porcentaje de salida guadianas recibidas", contenido_style))
    story.append(PageBreak())
    # Agregar secciones con anclas
    titulo_seccion_style = ParagraphStyle(
    'TituloSeccion',
    parent=styles['Normal'],
    fontSize=12,
    textColor=colors.white,  # Blanco
    alignment=1,  # Centrado
    fontName='Helvetica-Bold',  # Negrita
    spaceAfter=6  # Espacio después del título
    )

    for i, section_title in enumerate(sections_data, 1):
        print(i)
        # Crear ancla usando bookmarkPage
        anchor = f"section_{i}"
    
        # Añadir un espacio antes del cuadro del título
        story.append(Spacer(1, 70))
    
        # Crear la tabla para el título de la sección (cuadro con fondo)
        section_table = Table(
        [[Paragraph(f"<a name='{anchor}'/>{section_title}", titulo_seccion_style)]],  # Usamos contenido_style o un estilo específico si quieres
        colWidths=[6*inch]
        )
        section_table.setStyle(estilo_indice)  # Aplicamos el estilo del cuadro de descripción
        story.append(section_table)
        story.append(Spacer(1, 15))
        
        # Contenido de la sección debajo del cuadro
        story.append(Paragraph(f"Aquí va el contenido de la sección {section_title}.", titulo_seccion_style))
        story.append(Spacer(1, 10))
        origen = origenes[origenes["Subdirección"]== dirección[i-1]].copy()
        fig = gráfica_origenes(origen)
        if fig is not None:
            img_stream = BytesIO()
            write_image(fig, img_stream, format='png')
            img = Image(img_stream)
            img.drawHeight = 4 * inch  # Ajustar a tu necesidad
            img.drawWidth = 5 * inch   # Ajustar a tu necesidad
            story.append(img)
            story.append(Spacer(1, 12))
            table_data = []
            origen = origen.sort_values(by="Antelación", ascending=True)
            origen["Antelación"] = origen["Antelación"].apply(timedelta_to_hhmmss)
            # Encabezados de la tabla (puedes personalizarlos)
            table_data.append(['Código', 'Nombre', 'NTécnico','Producto','SalidaPlanificada','FechaSupresión','Antelación'])  # Ejemplo de encabezados

            # Llenar los datos de la tabla
            for index, row in origen.iterrows():
                table_data.append([row['Código'], row['Nombre'], row['NTécnico'],row['Producto'],row['SalidaPlanificada'],row['FechaSupresión'],row['Antelación']])  # Reemplaza con las columnas correctas
            # Crear la tabla con los datos
            table = Table(table_data)
            # Aplicar estilo a la tabla
            style = TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.grey),  # Color de fondo para los encabezados
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),  # Color de texto de los encabezados
                ('ALIGN', (0, 0), (-1, -1), 'CENTER'),  # Alinear texto al centro
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),  # Fuente en negrita para encabezados
                ('FONTSIZE', (0, 0), (-1, -1), 8),  # Reducir tamaño de la fuente
                ('BOTTOMPADDING', (0, 0), (-1, 0), 12),  # Espaciado en la parte inferior de los encabezados
                ('BACKGROUND', (0, 1), (-1, -1), colors.beige),  # Color de fondo para las filas
                ('GRID', (0, 0), (-1, -1), 1, colors.black),  # Agregar un borde a la tabla
            ])
            # Aplicar el estilo
            table.setStyle(style)
            # Añadir la tabla al documento
            story.append(table)
        else:
            story.append(Paragraph("No existen datos de supresiones de origen para la subdirección correspondiente.", contenido_style))

        ######## Cambio destino ########################################################
        destino = tabla2[tabla2["Subdirección"] == dirección[i-1]].copy()
        fig = grafica_destino(destino)
        if fig is not None:
            img_stream = BytesIO()
            write_image(fig, img_stream, format='png')
            img = Image(img_stream)
            img.drawHeight = 4 * inch  # Ajustar a tu necesidad
            img.drawWidth = 5 * inch   # Ajustar a tu necesidad
            story.append(img)
            story.append(Spacer(1, 12))
            table_data = []
            destino = destino.sort_values(by="Antelación", ascending=True)
            destino["Antelación"] = destino["Antelación"].apply(timedelta_to_hhmmss)
            # Encabezados de la tabla (puedes personalizarlos)
            table_data.append(['Código', 'Nombre', 'NTécnico','Producto','SalidaPlanificada','HoraAviso','HoraSupresión','Antelación'])  # Ejemplo de encabezados
            # Llenar los datos de la tabla
            for index, row in destino.iterrows():
                table_data.append([row['Código'], row['Nombre'], row['NTécnico'],row['Producto'],row['SalidaPlanificada'],row['HoraAviso'],row['HoraSupresión'],row['Antelación']])  # Reemplaza con las columnas correctas
            # Crear la tabla con los datos
            table = Table(table_data)
            # Aplicar estilo a la tabla
            style = TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.grey),  # Color de fondo para los encabezados
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),  # Color de texto de los encabezados
                ('ALIGN', (0, 0), (-1, -1), 'CENTER'),  # Alinear texto al centro
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),  # Fuente en negrita para encabezados
                ('FONTSIZE', (0, 0), (-1, -1), 6),  # Reducir tamaño de la fuente
                ('BOTTOMPADDING', (0, 0), (-1, 0), 12),  # Espaciado en la parte inferior de los encabezados
                ('BACKGROUND', (0, 1), (-1, -1), colors.beige),  # Color de fondo para las filas
                ('GRID', (0, 0), (-1, -1), 1, colors.black),  # Agregar un borde a la tabla
            ])
            # Aplicar el estilo
            table.setStyle(style)
            # Añadir la tabla al documento
            story.append(table)
        else:
            # Si no hay datos, mostrar un mensaje adecuado
            story.append(Paragraph("No existen datos de cambio de destino para la subdirección correspondiente.", contenido_style))
        ###################Guadiana######################################################################################################3
        guadiana=Guadiana1[Guadiana1["Subdirección"] ==  dirección[i-1]].copy()
        fig = gráficaGuadiana(guadiana)
        if fig is not None:
            img_stream = BytesIO()
            write_image(fig, img_stream, format='png')
            img = Image(img_stream)
            img.drawHeight = 4 * inch  # Ajustar a tu necesidad
            img.drawWidth = 5 * inch   # Ajustar a tu necesidad
            story.append(img)
            story.append(Spacer(1, 12))
            table_data = []
            # Encabezados de la tabla (puedes personalizarlos)
            table_data.append(['Código','Nombre','NTécnico',"Producto","Incidencia","TieneGuadiana"])  # Ejemplo de encabezados
            # Llenar los datos de la tabla
            for index, row in guadiana.iterrows():
                table_data.append([row['Código'], row['Nombre'], row['NTécnico'],row['Producto'],row['Incidencia'],row['TieneGuadiana']])  
            # Crear la tabla con los datos
            table = Table(table_data)
            # Aplicar estilo a la tabla
            style = TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.grey),  # Color de fondo para los encabezados
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),  # Color de texto de los encabezados
                ('ALIGN', (0, 0), (-1, -1), 'CENTER'),  # Alinear texto al centro
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),  # Fuente en negrita para encabezados
                ('FONTSIZE', (0, 0), (-1, -1), 8),  # Reducir tamaño de la fuente
                ('BOTTOMPADDING', (0, 0), (-1, 0), 12),  # Espaciado en la parte inferior de los encabezados
                ('BACKGROUND', (0, 1), (-1, -1), colors.beige),  # Color de fondo para las filas
                ('GRID', (0, 0), (-1, -1), 1, colors.black),  # Agregar un borde a la tabla
            ])
            # Aplicar el estilo
            table.setStyle(style)
            # Añadir la tabla al documento
            story.append(table)
        else:
            # Si no hay datos, mostrar un mensaje adecuado
            story.append(Paragraph("No existen datos de cambio de origen para la subdirección correspondiente.", contenido_style))

        
        # Salto de página excepto en la última sección
        if i < len(sections_data):
            story.append(PageBreak())

    # Construcción final del PDF
    doc.build(story, onFirstPage=add_header_footer, onLaterPages=add_header_footer)
    print(f"PDF con encabezado en tabla creado: {filename}")


In [ ]:
Supresiones_PDF()

In [ ]:
dirección[0]

In [ ]:
origenes[origenes["Subdirección"] == dirección[1]]